# 00 — Unduh Dataset CIC-IDS2017 (versi GeneratedLabelledFlows)

**Tujuan:** mengunduh CIC-IDS2017 versi yang masih memuat kolom **Source IP** dan **Timestamp**.
Kedua kolom ini dibutuhkan untuk menghitung fitur agregasi jendela waktu (misalnya
"berapa koneksi dari IP ini dalam 5 menit terakhir").

| | Versi lama (v1) | Versi ini (v2) |
|---|---|---|
| Nama | MachineLearningCVE | GeneratedLabelledFlows |
| Isi aliran & label | sama | sama |
| Source IP, Timestamp | **tidak ada** | **ada** |

- **Sumber asli:** Canadian Institute for Cybersecurity, UNB — https://www.unb.ca/cic/datasets/ids-2017.html (unduhan resmi lewat formulir).
- **Mirror:** Hugging Face `bencorn/CICIDS2017`, berkas `csvs/GeneratedLabelledFlows.zip` (nama dan isi sama dengan arsip resmi).
- **Integritas** diperiksa dengan SHA-256; **keaslian isi** diperiksa di notebook 01 dengan mencocokkan setiap baris ke versi v1.
- **Sitasi wajib:** Sharafaldin, Lashkari, & Ghorbani (2018), ICISSP, 108–116.

Cukup dijalankan **sekali**. Bila berkas sudah ada dan checksum cocok, unduhan dilewati.

In [ ]:
import os, sys

# Lokasi kerja: Google Colab (Drive) atau laptop
try:
    from google.colab import drive
    drive.mount('/content/drive')
    FOLDER = '/content/drive/MyDrive/TA/CIC-IDS2017'          # folder notebook ini di Google Drive
    CALON_KODE = [os.path.join(FOLDER, '2_sistem_nids'),      # lokasi folder kode sistem NIDS yang dicari
                  '/content/drive/MyDrive/TA/2_sistem_nids',
                  '/content/drive/MyDrive/TA/nids-deteksi',
                  '/content/drive/MyDrive/TA/04_Kode/nids-deteksi']
except ImportError:
    FOLDER = os.path.abspath(os.path.join('..', '..', '05_Data_Eksperimen', 'CIC-IDS2017'))  # folder data (notebook ada di 04_Kode/4_notebook)
    CALON_KODE = [os.path.abspath(os.path.join('..', '2_sistem_nids'))]                       # kode sistem NIDS

KODE = next((p for p in CALON_KODE if os.path.isdir(os.path.join(p, 'nids'))), None)
os.makedirs(FOLDER, exist_ok=True)
os.chdir(FOLDER)
DATA = os.path.join(FOLDER, 'data')
print('Folder kerja :', FOLDER)
print('Kode nids    :', KODE or 'tidak ditemukan (hanya dibutuhkan notebook 01)')

In [ ]:
import hashlib, json, urllib.request, zipfile

REPO, NAMA = 'bencorn/CICIDS2017', 'csvs/GeneratedLabelledFlows.zip'
ZIP = os.path.join(DATA, 'GeneratedLabelledFlows.zip')
OUT = os.path.join(DATA, 'raw_labelled')
os.makedirs(OUT, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blok in iter(lambda: f.read(1 << 20), b''):
            h.update(blok)
    return h.hexdigest()

info = json.load(urllib.request.urlopen(
    'https://huggingface.co/api/datasets/%s/tree/main/csvs' % REPO, timeout=60))
meta = next(f for f in info if f['path'] == NAMA)
expect, size = meta['lfs']['oid'], meta['size']

if not (os.path.exists(ZIP) and os.path.getsize(ZIP) == size and sha256(ZIP) == expect):
    print('mengunduh %s (%.1f MB) ...' % (NAMA, size / 1048576))
    urllib.request.urlretrieve('https://huggingface.co/datasets/%s/resolve/main/%s' % (REPO, NAMA), ZIP)
got = sha256(ZIP)
print('SHA-256', got, '->', 'OK' if got == expect else 'TIDAK COCOK')
assert got == expect, 'Unduhan rusak: hapus berkas zip lalu jalankan ulang.'

### Ekstrak berkas CSV
Hanya berkas `.csv` yang diambil, langsung ke `data/raw_labelled/` (tanpa subfolder arsip).

In [ ]:
with zipfile.ZipFile(ZIP) as z:
    for m in z.infolist():
        nama = os.path.basename(m.filename)
        if m.is_dir() or not nama.lower().endswith('.csv'):
            continue
        tujuan = os.path.join(OUT, nama)
        if not os.path.exists(tujuan) or os.path.getsize(tujuan) != m.file_size:
            with z.open(m) as src, open(tujuan, 'wb') as dst:
                dst.write(src.read())
        print('  %-62s %7.1f MB' % (nama, m.file_size / 1048576))
print('Selesai ->', OUT)